# Polar MHW Heat Budget: 6-Box Zonal Transect

Runs the same onset/decline budget decomposition as notebook 01, but across **6 adjacent
10×10 grid-cell boxes** (2.5°×2.5° each) arranged as a **zonal transect** at 66.5°S–64°S,
stepping westward from 70°W to 85°W (Drake Passage into the Bellingshausen Sea).

SST and budget data are loaded once for the full bounding region, then each box is
selected and processed in a loop. Season is assigned by the month of the event peak
(DJF/MAM/JJA/SON).

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import pandas as pd
import cmocean.cm as cmo
import cartopy.crs as ccrs
import cartopy.feature as cfeature

%matplotlib inline

In [ ]:
from dask.distributed import Client
client = Client()
client

## 1. Configuration

In [ ]:
def map_lon_to_ds(lon_e, lon_coord_min=-280.0, lon_coord_max=80.0):
    """Map standard longitudes to the ACCESS-OM2 grid convention."""
    lonmin, lonmax = lon_coord_min, lon_coord_max
    if 0.0 <= lonmin and lonmax <= 360.0:
        return lon_e % 360.0
    if -180.0 <= lonmin and lonmax <= 180.0:
        return ((lon_e + 180.0) % 360.0) - 180.0
    if lonmax <= 100.0 and lonmin < -180.0:
        return lon_e if lon_e <= lonmax else lon_e - 360.0
    return lon_e


def generate_boxes(lat_min, lat_max, lon_min, lon_max, box_deg=2.5):
    """
    Generate a regular grid of boxes.

    Longitudes are given in standard degrees (positive east); they are
    converted to the ACCESS-OM2 grid convention via map_lon_to_ds.
    For a circumpolar domain use lon_min=80, lon_max=440 (i.e. 80E eastward
    all the way round) — the dateline wrap is handled by map_lon_to_ds.

    Returns an OrderedDict {label: [lon_min_ds, lon_max_ds, lat_min, lat_max]}.
    """
    import numpy as np
    boxes = {}
    lat = lat_min
    while round(lat, 6) < lat_max:
        lon = lon_min
        while round(lon, 6) < lon_max:
            label = f"{abs(lat):.1f}–{abs(lat+box_deg):.1f}°S / {abs(lon):.1f}–{abs(lon+box_deg):.1f}°W"
            boxes[label] = [
                map_lon_to_ds(lon + 360),
                map_lon_to_ds(lon + box_deg + 360),
                lat,
                lat + box_deg,
            ]
            lon = round(lon + box_deg, 6)
        lat = round(lat + box_deg, 6)
    return boxes


# ── Box definition ────────────────────────────────────────────────────────────
# Current: 6-box zonal transect at 66.5°S–64°S stepping westward from 70°W to 85°W.
# To switch to circumpolar, replace with:
#   boxes = generate_boxes(lat_min=-80, lat_max=-60, lon_min=80, lon_max=440)
boxes = {
    '72.5–70°W': [map_lon_to_ds(287.5), map_lon_to_ds(290.0), -66.5, -64.0],
    '75–72.5°W': [map_lon_to_ds(285.0), map_lon_to_ds(287.5), -66.5, -64.0],
    '77.5–75°W': [map_lon_to_ds(282.5), map_lon_to_ds(285.0), -66.5, -64.0],
    '80–77.5°W': [map_lon_to_ds(280.0), map_lon_to_ds(282.5), -66.5, -64.0],
    '82.5–80°W': [map_lon_to_ds(277.5), map_lon_to_ds(280.0), -66.5, -64.0],
    '85–82.5°W': [map_lon_to_ds(275.0), map_lon_to_ds(277.5), -66.5, -64.0],
}
box_names = list(boxes.keys())
colors6   = plt.cm.viridis(np.linspace(0.1, 0.9, len(boxes)))

# Bounding region covering all boxes — loaded once, subsetted per box in the loop
all_lons = [v[0] for v in boxes.values()] + [v[1] for v in boxes.values()]
all_lats = [v[2] for v in boxes.values()] + [v[3] for v in boxes.values()]
wide_reg = [min(all_lons), max(all_lons), min(all_lats), max(all_lats)]

# ── Data path
base = '/g/data/av17/access-nri/OM2/025deg_jra55_iaf_cycle6_online_mlt/'

START_OUTPUT = 357   # 2010
END_OUTPUT   = 366   # 2019
outputs = list(range(START_OUTPUT, END_OUTPUT + 1))

MIN_DUR     = 5
MAX_GAP     = 2
chunks2D    = {'time': 1, 'yt_ocean': 216, 'xt_ocean': 240}
SEC_PER_DAY = 86400.0

# Budget term labels → raw variable names in the NetCDF files
# 'Surface Flux' is a derived term (surface_flux + sw_pen), handled separately
TERM_MAP = {
    'MLT tendency'   : 'mlt_tendency',
    'Surface Flux'   : 'surf_to_ML',
    'Advection'      : 'advection',
    'Vertical mixing': 'vert_mixing',
    'Entrainment'    : 'entrainment',
}
BUDGET_TERMS = list(TERM_MAP.keys())

## 2. Context map — all 6 boxes

In [ ]:
fig = plt.figure(figsize=(7, 7))
ax  = plt.axes(projection=ccrs.SouthPolarStereo())
ax.set_extent([-180, 180, -90, -50], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor='0.85', edgecolor='k', linewidth=0.5, zorder=2)
ax.coastlines(linewidth=0.7, zorder=3)
ax.gridlines(draw_labels=False, linestyle=':', linewidth=0.4, alpha=0.5)

for (name, sreg), c in zip(boxes.items(), colors6):
    rect = Rectangle(
        (sreg[0], sreg[2]), sreg[1] - sreg[0], sreg[3] - sreg[2],
        facecolor=c, edgecolor='k', linewidth=1.0,
        alpha=0.7, transform=ccrs.PlateCarree(), zorder=4,
    )
    ax.add_patch(rect)

ax.legend(
    handles=[Rectangle((0,0),1,1, facecolor=c, alpha=0.7, label=n)
             for n, c in zip(box_names, colors6)],
    loc='lower left', fontsize=8, title='Box',
)
ax.set_title('6-box zonal transect (66.5°S–64°S, 85°W–70°W)', fontsize=12)
plt.tight_layout()

## 3. Load SST and budget data for the full bounding region (once)

In [ ]:
sst_files = [base + f'output{o:03d}/ocean/ocean_daily.nc' for o in outputs]

ds_sst = xr.open_mfdataset(
    sst_files,
    decode_times=True, chunks=chunks2D,
    combine='nested', concat_dim='time',
    data_vars=['temp_in_mld', 'mld'],
    parallel=True, decode_timedelta=False,
).sel(xt_ocean=slice(wide_reg[0], wide_reg[1]),
      yt_ocean=slice(wide_reg[2], wide_reg[3]))

temp_wide = ds_sst['temp_in_mld'] / 1035

print(f"SST loaded (lazy): {dict(ds_sst.dims)}")

In [ ]:
budget_files = [
    base + f'post_processed_diags/mlt_budget_online_stavg/mlt_budget_stavg_daily_online_output{o:03d}.nc'
    for o in outputs
]
vars_raw = ['mlt_tendency', 'advection', 'vert_mixing', 'entrainment', 'surface_flux', 'sw_pen', 'residual']

budget_wide = xr.open_mfdataset(
    budget_files,
    decode_times=True, chunks=chunks2D,
    combine='nested', concat_dim='time',
    parallel=True, decode_timedelta=False,
).sel(xt_ocean=slice(wide_reg[0], wide_reg[1]),
      yt_ocean=slice(wide_reg[2], wide_reg[3]))[vars_raw]

budget_wide['surf_to_ML'] = budget_wide['surface_flux'] + budget_wide['sw_pen']

print(f"Budget loaded (lazy): {dict(budget_wide.dims)}")

## 4. Load climatology, threshold, and budget climatology

In [ ]:
clim_wide   = xr.open_dataset(base + 'post_processed_diags/om2_025_MLT_clim.nc').sel(
    xt_ocean=slice(wide_reg[0], wide_reg[1]), yt_ocean=slice(wide_reg[2], wide_reg[3]))
thresh_wide = xr.open_dataset(base + 'post_processed_diags/om2_025_MLT_thresh.nc').sel(
    xt_ocean=slice(wide_reg[0], wide_reg[1]), yt_ocean=slice(wide_reg[2], wide_reg[3]))

budget_clim_wide = xr.open_dataset(
    base + 'post_processed_diags/mlt_budget_online_stavg/'
           'mlt_budget_stavg_daily_online_output336-365_monthly_mean.ncea.nc'
).sel(xt_ocean=slice(wide_reg[0], wide_reg[1]), yt_ocean=slice(wide_reg[2], wide_reg[3]))
budget_clim_wide['surf_to_ML'] = budget_clim_wide['surface_flux'] + budget_clim_wide['sw_pen']

print("Climatology files opened.")

## 5. Helper functions

In [ ]:
def detect_mhw_events(temp_da, thresh_da, min_dur=5, max_gap=2):
    """Hobday et al. (2016) event detection. Returns list of dicts: t_start, t_peak, t_end."""
    t      = pd.to_datetime(temp_da.time.values)
    y_temp = temp_da.values
    y_thr  = thresh_da.values
    valid  = np.isfinite(y_temp) & np.isfinite(y_thr)

    exceed = pd.Series((y_temp[valid] > y_thr[valid]).astype(int), index=t[valid])

    runs        = (exceed.diff(1).ne(0)).cumsum()
    run_lengths = exceed.groupby(runs).transform('size')
    short_false = (exceed == 0) & (run_lengths <= max_gap)
    merged      = exceed.copy()
    merged[short_false] = 1

    runs2     = (merged.diff(1).ne(0)).cumsum()
    lens2     = merged.groupby(runs2).transform('size')
    long_true = (merged == 1) & (lens2 >= min_dur)

    anom = pd.Series(y_temp[valid] - y_thr[valid], index=t[valid])
    arr, idx = long_true.to_numpy(), long_true.index.to_numpy()
    events, start_i = [], None
    for i in range(len(arr)):
        if arr[i] and start_i is None:
            start_i = i
        if start_i is not None and (i == len(arr) - 1 or not arr[i + 1]):
            seg    = anom.iloc[start_i:i + 1]
            t_peak = seg.idxmax()
            events.append({'t_start': idx[start_i], 't_peak': t_peak, 't_end': idx[i]})
            start_i = None
    return events


def get_season(timestamp):
    """Return austral season label based on month of timestamp."""
    m = pd.Timestamp(timestamp).month
    if m in (12, 1, 2):
        return 'DJF'
    elif m in (3, 4, 5):
        return 'MAM'
    elif m in (6, 7, 8):
        return 'JJA'
    else:
        return 'SON'


def build_budget_clim_daily(budget_clim_box, budget_time):
    """Interpolate monthly budget climatology to daily and apply 31-day smoothing."""
    clim_by_month = (
        budget_clim_box.groupby('time.month').mean('time')
        .assign_coords(month=np.arange(1, 13))
    )
    dec = clim_by_month.sel(month=12).expand_dims(month=[0])
    jan = clim_by_month.sel(month=1).expand_dims(month=[13])
    clim_padded = xr.concat([dec, clim_by_month, jan], dim='month')
    clim_smooth = clim_padded.interp(month=np.linspace(1, 12, 365))

    doys = (budget_time.dt.dayofyear - 1) % 365
    clim_daily = xr.Dataset({
        name: clim_smooth[name].isel(month=doys).assign_coords(time=budget_time)
        for name in BUDGET_TERMS
    })
    return clim_daily.rolling(time=31, center=True, min_periods=1).mean()


SEASONS      = ['DJF', 'MAM', 'JJA', 'SON']
SEASON_COLORS = {'DJF': 'tab:red', 'MAM': 'tab:orange', 'JJA': 'tab:blue', 'SON': 'tab:green'}

print("Helper functions defined.")

## 6. Pre-compute all box means — single Dask pass

Build one lazy expression per (box × quantity), collect into a flat Dataset,
then call `.compute()` once.  Dask executes the full graph in one optimised
pass over the data, regardless of how many boxes there are.

## 7. Loop over boxes — event detection and budget decomposition (in-memory)

In [ ]:
all_events = {}
all_df     = {}

for box_name in box_names:
    print(f"\n── {box_name} ──────────────────────────────")

    # Pull pre-computed in-memory time series
    temp_vals   = sst_all[box_name]
    clim_mean   = clim_all[box_name]
    thresh_mean = thresh_all[box_name]

    doy_all        = temp_vals.time.dt.dayofyear.clip(max=365)
    clim_on_time   = clim_mean.sel(dayofyear=doy_all).assign_coords(time=temp_vals.time)
    thresh_on_time = thresh_mean.sel(dayofyear=doy_all).assign_coords(time=temp_vals.time)

    events = detect_mhw_events(temp_vals, thresh_on_time, MIN_DUR, MAX_GAP)
    print(f"  {len(events)} MHW events detected")
    all_events[box_name] = events

    if not events:
        all_df[box_name] = pd.DataFrame()
        continue

    # Assemble budget series from pre-computed means
    budget_series = xr.Dataset({term: budget_all[f"{box_name}__{term}"]
                                 for term in BUDGET_TERMS})

    # Budget climatology → daily anomaly (pure numpy, no Dask)
    bc_box = xr.Dataset({term: bclim_all[f"{box_name}__{term}"]
                          for term in BUDGET_TERMS})
    clim_daily  = build_budget_clim_daily(bc_box, budget_series.time)
    anom_series = xr.Dataset({name: budget_series[name] - clim_daily[name]
                               for name in BUDGET_TERMS})

    # Onset / decline decomposition
    rows = []
    for ev in events:
        t_s, t_p, t_e = ev['t_start'], ev['t_peak'], ev['t_end']
        season = get_season(t_p)
        for name in BUDGET_TERMS:
            rows.append({
                't_start': t_s, 't_peak': t_p, 't_end': t_e,
                'season' : season,
                'term'   : name,
                'onset'  : float(anom_series[name].sel(time=slice(t_s, t_p)).mean()),
                'decline': float(anom_series[name].sel(time=slice(t_p, t_e)).mean()),
            })
    all_df[box_name] = pd.DataFrame(rows)

print("\nAll boxes processed.")

In [ ]:
def _weights(da, lat_coord='yt_ocean'):
    return np.cos(np.deg2rad(da[lat_coord]))

def _box_mean(da, sreg):
    sub = da.sel(xt_ocean=slice(sreg[0], sreg[1]),
                 yt_ocean=slice(sreg[2], sreg[3]))
    return sub.weighted(_weights(sub)).mean(('xt_ocean', 'yt_ocean'))


# ── SST and climatology means (one compute) ───────────────────────────────────
print("Building SST / clim / threshold lazy graph ...")
sst_lazy   = {name: _box_mean(temp_wide,       sreg) for name, sreg in boxes.items()}
clim_lazy  = {name: _box_mean(clim_wide.temp,  sreg) for name, sreg in boxes.items()}
thresh_lazy= {name: _box_mean(thresh_wide.temp,sreg) for name, sreg in boxes.items()}

print("Computing SST / clim / threshold for all boxes ...")
sst_all    = xr.Dataset(sst_lazy).compute()
clim_all   = xr.Dataset(clim_lazy).compute()
thresh_all = xr.Dataset(thresh_lazy).compute()
print("Done.")

# ── Budget means (one compute) ────────────────────────────────────────────────
# Flat variable names: "{box_name}__{term_label}"
print("\nBuilding budget lazy graph ...")
budget_lazy = {}
for box_name, sreg in boxes.items():
    braw = budget_wide.sel(xt_ocean=slice(sreg[0], sreg[1]),
                           yt_ocean=slice(sreg[2], sreg[3]))
    w = _weights(braw)
    for term, raw in TERM_MAP.items():
        budget_lazy[f"{box_name}__{term}"] = (
            braw[raw].weighted(w).mean(('yt_ocean', 'xt_ocean')) * SEC_PER_DAY
        )

print("Computing budget means for all boxes ...")
budget_all = xr.Dataset(budget_lazy).compute()
print("Done.")

# ── Budget climatology means (cheap — small monthly files) ────────────────────
print("\nComputing budget climatology means ...")
bclim_lazy = {}
for box_name, sreg in boxes.items():
    bc = budget_clim_wide.sel(xt_ocean=slice(sreg[0], sreg[1]),
                              yt_ocean=slice(sreg[2], sreg[3]))
    w = _weights(bc)
    for term, raw in TERM_MAP.items():
        bclim_lazy[f"{box_name}__{term}"] = (
            bc[raw].weighted(w).mean(('yt_ocean', 'xt_ocean')) * SEC_PER_DAY
        )
bclim_all = xr.Dataset(bclim_lazy).compute()
print("Done — all data in memory.")

## 8. Event summary table

In [ ]:
summary_rows = []
for name, evs in all_events.items():
    if not evs:
        summary_rows.append({'box': name, 'n_events': 0,
                             'mean_duration': np.nan,
                             'mean_onset_days': np.nan,
                             'mean_decline_days': np.nan})
        continue
    durs = [(pd.Timestamp(e['t_end']) - pd.Timestamp(e['t_start'])).days for e in evs]
    on_d = [(pd.Timestamp(e['t_peak']) - pd.Timestamp(e['t_start'])).days for e in evs]
    de_d = [(pd.Timestamp(e['t_end'])  - pd.Timestamp(e['t_peak'])).days  for e in evs]
    summary_rows.append({
        'box'              : name,
        'n_events'         : len(evs),
        'mean_duration'    : np.mean(durs),
        'mean_onset_days'  : np.mean(on_d),
        'mean_decline_days': np.mean(de_d),
    })

summary = pd.DataFrame(summary_rows).set_index('box')
print(summary.round(1))

## 9. Plots

### 8a. Onset and decline composites — grouped bar charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

x     = np.arange(len(BUDGET_TERMS))
n     = len(boxes)
width = 0.8 / n

for phase, ax in zip(['onset', 'decline'], axes):
    for i, (box_name, c) in enumerate(zip(box_names, colors6)):
        df = all_df[box_name]
        if df.empty:
            continue
        means  = df.groupby('term')[phase].mean().reindex(BUDGET_TERMS)
        offset = (i - n/2 + 0.5) * width
        ax.bar(x + offset, means, width * 0.9,
               label=box_name, color=c, alpha=0.85)

    ax.axhline(0, color='k', linewidth=0.8, alpha=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([t.replace(' ', '\n') for t in BUDGET_TERMS], fontsize=10)
    ax.set_title(phase.capitalize(), fontsize=12)
    ax.grid(True, linestyle=':', alpha=0.5, axis='y')

axes[0].set_ylabel('Budget anomaly (°C day⁻¹)', fontsize=11)
axes[1].legend(title='Box (lat)', fontsize=9, ncol=2, loc='upper right')
fig.suptitle('Mean budget anomaly by box and phase (2010–2019)', fontsize=13)
plt.tight_layout()

### 8b. Heatmaps — onset vs decline by box and term

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, phase in zip(axes, ['onset', 'decline']):
    mat = np.full((len(box_names), len(BUDGET_TERMS)), np.nan)
    for r, box_name in enumerate(box_names):
        df = all_df[box_name]
        if df.empty:
            continue
        for c, term in enumerate(BUDGET_TERMS):
            mat[r, c] = df[df.term == term][phase].mean()

    vmax = np.nanmax(np.abs(mat))
    im   = ax.imshow(mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(BUDGET_TERMS)))
    ax.set_xticklabels([t.replace(' ', '\n') for t in BUDGET_TERMS], fontsize=9)
    ax.set_yticks(range(len(box_names)))
    ax.set_yticklabels(box_names, fontsize=9)
    ax.set_title(phase.capitalize(), fontsize=12)
    plt.colorbar(im, ax=ax, label='°C day⁻¹', shrink=0.85)

fig.suptitle('Mean budget anomaly heatmap — 6-box transect (2010–2019)', fontsize=13)
plt.tight_layout()

### 8c. Event count and duration by box

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(box_names, summary['n_events'],
            color=colors6, alpha=0.85, edgecolor='k', linewidth=0.5)
axes[0].set_ylabel('Number of events')
axes[0].set_title('MHW event count per box')
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(True, linestyle=':', alpha=0.4, axis='y')

axes[1].bar(np.arange(len(box_names)) - 0.2, summary['mean_onset_days'],   0.35,
            color=colors6, alpha=0.85, edgecolor='k', linewidth=0.5, label='Onset')
axes[1].bar(np.arange(len(box_names)) + 0.2, summary['mean_decline_days'], 0.35,
            color=colors6, alpha=0.5,  edgecolor='k', linewidth=0.5, label='Decline',
            hatch='//')
axes[1].set_xticks(range(len(box_names)))
axes[1].set_xticklabels(box_names, rotation=30)
axes[1].set_ylabel('Mean days')
axes[1].set_title('Mean onset and decline duration per box')
axes[1].legend()
axes[1].grid(True, linestyle=':', alpha=0.4, axis='y')

plt.tight_layout()

## 10. Seasonal breakdown

Season is assigned by the month of `t_peak` (day of maximum temperature anomaly):
- **DJF** — austral summer
- **MAM** — austral autumn
- **JJA** — austral winter
- **SON** — austral spring

In [ ]:
# Season counts across all boxes
all_combined = pd.concat(
    [df.assign(box=name) for name, df in all_df.items() if not df.empty],
    ignore_index=True,
)

# One row per event (not per term) for counting
ev_rows = all_combined.drop_duplicates(subset=['box','t_start','t_peak','t_end'])
print("Events per season across all boxes:")
print(ev_rows.groupby('season')['t_start'].count().reindex(SEASONS).fillna(0).astype(int))

### 9a. Seasonal composites — all boxes combined

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

x      = np.arange(len(BUDGET_TERMS))
n_seas = len(SEASONS)
width  = 0.8 / n_seas

for phase, ax in zip(['onset', 'decline'], axes):
    for i, season in enumerate(SEASONS):
        sub   = all_combined[all_combined.season == season]
        means = sub.groupby('term')[phase].mean().reindex(BUDGET_TERMS)
        offset = (i - n_seas/2 + 0.5) * width
        ax.bar(x + offset, means, width * 0.9,
               label=season, color=SEASON_COLORS[season], alpha=0.85)

    ax.axhline(0, color='k', linewidth=0.8, alpha=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([t.replace(' ', '\n') for t in BUDGET_TERMS], fontsize=10)
    ax.set_title(phase.capitalize(), fontsize=12)
    ax.grid(True, linestyle=':', alpha=0.5, axis='y')

axes[0].set_ylabel('Budget anomaly (°C day⁻¹)', fontsize=11)
axes[1].legend(title='Season', fontsize=10)
fig.suptitle('Seasonal budget anomaly composites — all 6 boxes combined (2010–2019)', fontsize=13)
plt.tight_layout()

### 9b. Seasonal heatmaps — one panel per season, onset vs decline side by side

In [ ]:
fig, axes = plt.subplots(len(SEASONS), 2,
                         figsize=(13, 3.2 * len(SEASONS)),
                         sharey='row', sharex='col')

# Global colour scale across all seasons and phases
all_vals = []
for season in SEASONS:
    sub = all_combined[all_combined.season == season]
    for phase in ['onset', 'decline']:
        mat = np.full((len(box_names), len(BUDGET_TERMS)), np.nan)
        for r, bn in enumerate(box_names):
            s2 = sub[sub.box == bn]
            for c, term in enumerate(BUDGET_TERMS):
                vals = s2[s2.term == term][phase]
                if len(vals):
                    mat[r, c] = vals.mean()
        all_vals.append(mat)
vmax = np.nanmax(np.abs(np.concatenate([m.ravel() for m in all_vals])))

for row, season in enumerate(SEASONS):
    sub = all_combined[all_combined.season == season]
    for col, phase in enumerate(['onset', 'decline']):
        ax  = axes[row, col]
        mat = np.full((len(box_names), len(BUDGET_TERMS)), np.nan)
        for r, bn in enumerate(box_names):
            s2 = sub[sub.box == bn]
            for c, term in enumerate(BUDGET_TERMS):
                vals = s2[s2.term == term][phase]
                if len(vals):
                    mat[r, c] = vals.mean()

        n_ev = len(sub.drop_duplicates(subset=['box','t_start','t_peak','t_end']))
        im   = ax.imshow(mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')

        ax.set_xticks(range(len(BUDGET_TERMS)))
        ax.set_yticks(range(len(box_names)))
        if row == len(SEASONS) - 1:
            ax.set_xticklabels([t.replace(' ', '\n') for t in BUDGET_TERMS], fontsize=8)
        else:
            ax.set_xticklabels([])
        if col == 0:
            ax.set_yticklabels(box_names, fontsize=8)
            ax.set_ylabel(f'{season}\n({n_ev} events)', fontsize=9, fontweight='bold')
        else:
            ax.set_yticklabels([])

        ax.set_title(phase.capitalize() if row == 0 else '', fontsize=11)

        # Annotate cells with NaN (no events) with an 'x'
        for r in range(len(box_names)):
            for c in range(len(BUDGET_TERMS)):
                if np.isnan(mat[r, c]):
                    ax.text(c, r, '×', ha='center', va='center', fontsize=9, color='0.5')

fig.colorbar(im, ax=axes, label='°C day⁻¹', shrink=0.6, pad=0.02)
fig.suptitle('Budget anomaly by season, box and phase (2010–2019)', fontsize=13, y=1.01)
plt.tight_layout()